# Demo GSTL (risiko tabrakan dari video dashcam) di Google Colab

Notebook ini menjalankan `demo.py` dari repo `gstl_replication` pada **video Anda sendiri**: skor risiko per frame, grafik, dan video overlay.

**Sebelum mulai**
1. Menu *Runtime → Change runtime type → T4 GPU*.
2. Repo privat? Simpan token GitHub (fine-grained, hanya baca repo ini) di *Secrets* (ikon kunci) dengan nama `GITHUB_TOKEN`, lalu aktifkan *Notebook access*. Jangan menulis token di sel.
3. Siapkan bobot detektor `latest.pth` (969 MB) di Google Drive Anda (disarankan) atau biarkan sel 4 mengunduh dari Drive penulis UString.

**Keterbatasan** (jangan diklaim lebih): model dilatih hanya pada CCD 1280x720; video lain di luar distribusi latih; skor bukan risiko terkalibrasi; detektor tanpa kelas motor. Notebook ini belum teruji di semua versi Colab.

In [ ]:
# 1. Periksa GPU dan versi
import sys, torch, numpy
print("python", sys.version.split()[0], "| torch", torch.__version__, "| numpy", numpy.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA (demo tetap jalan di CPU, jauh lebih lambat)")

In [ ]:
# 2. Ambil kode dari GitHub (token dibaca dari Secrets bila ada; tidak pernah dicetak)
import os, subprocess
REPO = "taufiqmus/gstl_replication"
if not os.path.isdir("/content/gstl_replication"):
    try:
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
    except Exception:
        tok = None
    url = ("https://%s@github.com/%s.git" % (tok, REPO)) if tok else ("https://github.com/%s.git" % REPO)
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", url, "/content/gstl_replication"], capture_output=True, text=True)
    if r.returncode:
        raise SystemExit("clone gagal (repo privat tanpa token, atau token salah): " + r.stderr.replace(tok or "<none>", "***"))
    if tok:  # hapus token dari konfigurasi git di mesin Colab
        subprocess.run(["git", "-C", "/content/gstl_replication", "remote", "set-url", "origin", "https://github.com/%s.git" % REPO])
%cd /content/gstl_replication
!git log --oneline -1

In [ ]:
# 3. Pasang dependensi demo (torch dan numpy bawaan Colab TIDAK diganti)
!pip -q install "opencv-python-headless" "Pillow" "matplotlib"
import cv2; print("opencv", cv2.__version__)

In [ ]:
# 4. Bobot detektor Cascade R-CNN (milik penulis UString, tidak ada di repo)
import shutil, glob
DET = "/content/gstl_replication/weights/detector/latest.pth"
os.makedirs(os.path.dirname(DET), exist_ok=True)

# Pilihan A (disarankan): salinan di Google Drive Anda. Ubah path bila perlu, atau kosongkan.
DRIVE_PATH = "/content/drive/MyDrive/gstl/latest.pth"
if not os.path.exists(DET) and DRIVE_PATH:
    from google.colab import drive
    drive.mount("/content/drive")
    if os.path.exists(DRIVE_PATH):
        shutil.copy(DRIVE_PATH, DET)

# Pilihan B: unduh dari folder Drive penulis UString (lambat, bisa kena batas kuota unduhan Drive)
if not os.path.exists(DET):
    !pip -q install gdown
    !gdown --folder "https://drive.google.com/drive/folders/1vUz3HYoAFeu0qGQy7pe51oa41zd0OCQU" -O /content/ustring_drive
    found = glob.glob("/content/ustring_drive/**/latest.pth", recursive=True)
    if not found:
        raise SystemExit("latest.pth tidak ditemukan; unduh manual dan taruh di %s" % DRIVE_PATH)
    shutil.copy(found[0], DET)

# Periksa hash SEBELUM dipakai (berkas dimuat dengan unpickle penuh)
!python tools/check_setup.py --detector "{DET}" --hash-detector

In [ ]:
# 5. Unggah video (mp4/avi/mov). Atau isi VIDEO dengan path di Drive/Colab dan lewati unggahan.
VIDEO = ""
if not VIDEO:
    from google.colab import files
    up = files.upload()
    VIDEO = "/content/" + next(iter(up))
    os.replace(next(iter(up)), VIDEO)
print("video:", VIDEO)

In [ ]:
# 6. Jalankan demo. MAX_WINDOWS=0 memproses seluruh video; isi mis. 4 untuk uji cepat (4 jendela x 50 frame).
MAX_WINDOWS = 0
OUT = "/content/hasil_demo"
!python -W ignore demo.py --video "{VIDEO}" --detector "{DET}" --out "{OUT}" --viz full --uncertainty --html --threshold 0.5 --width 960 --max-windows {MAX_WINDOWS}

In [ ]:
# 7. Tampilkan grafik skor dan video overlay
from IPython.display import Image, display, HTML
from base64 import b64encode
print(sorted(os.listdir(OUT)))
display(Image(OUT + "/skor.png"))
if os.path.exists(OUT + "/overlay.mp4"):
    data = b64encode(open(OUT + "/overlay.mp4", "rb").read()).decode()
    display(HTML('<video width="720" controls src="data:video/mp4;base64,%s"></video>' % data))

In [ ]:
# 8. Simpan hasil (zip) agar tidak hilang saat sesi berakhir
from google.colab import files
shutil.make_archive("/content/hasil_demo", "zip", OUT)
files.download("/content/hasil_demo.zip")

## Catatan
- Skor dihitung per jendela 50 frame (keadaan GRU di-reset), jadi skor rendah pada ±1 detik pertama tiap jendela itu **artefak**, bukan perubahan risiko.
- Bila `latest.pth` tidak tersedia, `demo.py` punya `--det-model tv_fasterrcnn` (bobot COCO, diunduh otomatis), tetapi itu bukan pipeline CCD asli.
- Video overlay panjang bisa besar; untuk uji cepat pakai `MAX_WINDOWS = 4`.